In [1]:
import langchain
import langchain_core

print("LangChain:", langchain.__version__)
print("LangChain Core:", langchain_core.__version__)


LangChain: 1.2.12
LangChain Core: 1.2.18


In [2]:
from langchain_core.tools import tool

@tool
def calculate_effective_rainfall(
    precipitation_mm: float,
    infiltration_mm: float
) -> float:
    """
    计算有效降雨量。

    Args:
        precipitation_mm: 总降水量，单位mm。
        infiltration_mm: 入渗量，单位mm。

    Returns:
        有效降雨量，单位mm。
    """

    if precipitation_mm < 0 or infiltration_mm < 0:
        raise ValueError("降水量和入渗量不能为负数")

    if infiltration_mm > precipitation_mm:
        raise ValueError("入渗量不能大于总降水量")

    return precipitation_mm - infiltration_mm

tools = [
    calculate_effective_rainfall
]

for t in tools:
    print(t.name)
    print(t.description)

calculate_effective_rainfall
计算有效降雨量。

Args:
    precipitation_mm: 总降水量，单位mm。
    infiltration_mm: 入渗量，单位mm。

Returns:
    有效降雨量，单位mm。


In [4]:
# 调用 LLM 生成答案
import os
from langchain_deepseek import ChatDeepSeek
from langchain_community.chat_models import ChatTongyi
from langchain_openai import ChatOpenAI
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from dotenv import load_dotenv
from rich import print as rprint
import time


load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

llmodel_deepseek = init_chat_model(
    model = "deepseek-flash",
    api_key = DEEPSEEK_API_KEY,
    api_base = DEEPSEEK_BASE_URL,

)



In [5]:
from langchain_core.messages import (
    SystemMessage,
    HumanMessage
)

agent_model = llmodel_deepseek.bind_tools(tools)

response = agent_model.invoke(
    [
        SystemMessage(
            content="你是一个水文气象分析助手。"
                    "当用户需要计算有效降雨量时，"
                    "优先使用提供的计算工具。"
        ),
        HumanMessage(
            content="某次降雨总量为120mm，"
                    "其中入渗量为35mm，"
                    "请计算有效降雨量。"
        )
    ],
    temperature=0
)

print(response)
print("Tool calls:")

for tool_call in response.tool_calls:
    print("Name:", tool_call["name"])
    print("Args:", tool_call["args"])
    print("ID:", tool_call["id"])

content='我来为您计算有效降雨量。' additional_kwargs={'refusal': None, 'reasoning_content': 'The user wants effective rainfall = precipitation - infiltration = 120 - 35 = 85 mm. Use the tool.'} response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 405, 'total_tokens': 499, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 24, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 405}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '2d56dca1-dfd3-4fb1-b20f-9e386f68929b', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--01a1d503-1f08-772c-86a7-639182993115-0' tool_calls=[{'name': 'calculate_effective_rainfall', 'args': {'precipitation_mm': 120, 'infiltration_mm': 35}, 'id': 'call_00_vrmtSo75fP4mGbQ2ztHW4067', 'type': 'tool

In [6]:
from langchain_core.messages import ToolMessage

# 建立工具名称与工具对象的映射
tool_map = {
    tool.name: tool
    for tool in tools
}

# 获取模型生成的工具调用
for tool_call in response.tool_calls:

    tool_name = tool_call["name"]
    tool_args = tool_call["args"]

    # 找到对应工具
    selected_tool = tool_map[tool_name]

    # 执行工具
    tool_result = selected_tool.invoke(tool_args)

    print("工具名称:", tool_name)
    print("工具参数:", tool_args)
    print("执行结果:", tool_result)

工具名称: calculate_effective_rainfall
工具参数: {'precipitation_mm': 120, 'infiltration_mm': 35}
执行结果: 85.0


In [7]:
messages = [
    SystemMessage(
        content="你是一个专业的水文气象分析助手。"
    ),
    HumanMessage(
        content="某次降雨总量为120mm，其中入渗量为35mm，请计算有效降雨量。"
    ),
    response
]
for tool_call in response.tool_calls:

    selected_tool = tool_map[tool_call["name"]]

    tool_result = selected_tool.invoke(
        tool_call["args"]
    )

    tool_message = ToolMessage(
        content=str(tool_result),
        tool_call_id=tool_call["id"]
    )

    messages.append(tool_message)

final_response = llmodel_deepseek.invoke(
    messages,
    temperature=0
)

print(final_response.content)

## 计算结果

**有效降雨量 = 85 mm**

### 计算过程

有效降雨量是指降雨总量中扣除被土壤入渗吸收后、能够形成地表径流的那部分降雨，公式为：

$$P_{eff} = P - F$$

其中：
- $P$ 为降雨总量 = 120 mm
- $F$ 为入渗量 = 35 mm

$$P_{eff} = 120 - 35 = 85\ \text{mm}$$

### 说明

在这次降雨事件中，约 **29.2%** 的降雨量（35 mm）下渗进入土壤，剩余 **70.8%**（85 mm）的降雨会留存于地表，可能形成地表径流或地面积水。需要注意的是，该计算为简化的一步扣除法，未考虑蒸发、填洼、植物截留等损失项；若需要更精确的产流估算，还应结合下垫面条件与初损参数进一步修正。


In [8]:
def run_agent(query, tools, max_iterations=5):

    tool_map = {
        tool.name: tool
        for tool in tools
    }

    messages = [
        SystemMessage(
            content="你是一个专业的水文气象分析助手。"
                    "需要计算时，优先调用提供的工具。"
        ),
        HumanMessage(content=query)
    ]

    model_with_tools = llmodel_deepseek.bind_tools(tools)

    for step in range(max_iterations):

        # 1. 模型决策
        response = model_with_tools.invoke(
            messages,
            temperature=0
        )

        messages.append(response)

        # 2. 没有工具调用，返回最终答案
        if not response.tool_calls:
            return response.content

        # 3. 执行工具
        for tool_call in response.tool_calls:

            tool_name = tool_call["name"]

            if tool_name not in tool_map:
                raise ValueError(
                    f"未知工具: {tool_name}"
                )

            selected_tool = tool_map[tool_name]

            tool_result = selected_tool.invoke(
                tool_call["args"]
            )

            # 4. 工具结果回传
            messages.append(
                ToolMessage(
                    content=str(tool_result),
                    tool_call_id=tool_call["id"]
                )
            )

    return "达到最大工具调用次数，任务未完成。"

In [9]:
result = run_agent(
    query="某次降雨总量为120mm，其中入渗量为35mm，请计算有效降雨量，并解释计算结果。",
    tools=tools
)

print(result)

## 计算结果

**有效降雨量 = 85.0 mm**

## 计算说明

- 降雨总量：120 mm
- 入渗量：35 mm
- 有效降雨量 = 降雨总量 − 入渗量 = 120 − 35 = **85 mm**
- 产流系数（径流系数）≈ 85 / 120 ≈ **0.71**（约 71%）

## 结果解释

1. **物理含义**：有效降雨量指扣除被土壤吸收、下渗到包气带或地下水的那部分水量后，真正能够形成地表径流、参与坡面汇流并汇入河网的那部分降水量。本次降雨中约有 85 mm 将转化为地表径流，35 mm 则被下渗、土壤蓄持和植物截留等作用消耗。

2. **水文意义**：0.71 的产流系数属于偏高的量级，说明下垫面下渗能力相对有限（如土壤较饱和、前期降雨充沛、植被覆盖较差或存在不透水面积等），降雨—径流响应较为迅速，洪水风险相对较高，需关注中小河流洪水与城市内涝。

3. **方法局限**：该计算为**初损后损法/简化水量平衡**的粗略估算，仅作总量上的扣除，未考虑：
   - **蒸发与蒸散发**、植物截留、填洼等损失项；
   - **降雨强度与时程分布**（短历时高强度降雨的产流效率通常更高）；
   - **前期土壤含水量**变化导致的入渗能力时变特征；
   - **下渗量的空间不均匀性**。

   若用于洪水预报或工程设计，建议改用 SCS-CN 法、下渗曲线法（如 Horton、Green-Ampt）或分布式水文模型，并结合实测径流资料率定参数。

如需，我可以进一步按 SCS-CN 方法估算本次降雨的径流量，或分析不同前期湿润条件下的产流差异。
